In [7]:
import pandas as pd 
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# --- CONFIGURAÇÃO ---
years = range(2015, 2025)
ks = [1, 10, 30]
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]

# Função que calcula o beta
def beta_ols(rp, rm):
    # Garante que rm seja uma Series com nome
    rm = rm.squeeze().rename("rm")
    df = rp.to_frame("rp").join(rm, how="inner").dropna()
    
    if len(df) < 5: return np.nan # Proteção contra dados insuficientes
    
    y = df["rp"]
    X = sm.add_constant(df["rm"])
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5}
    )
    return model.params["rm"]

# Dicionário para armazenar os betas finais ou retornos
# Estrutura: results[year][k][centrality]
all_betas = {}

for year in years:
    print(f"Processando ano: {year}...")
    all_betas[year] = {}
    
    # 1. Baixando Benchmark (SP500) para o ano específico
    # Usamos start/end cobrindo o ano para bater com os retornos dos portfolios
    sp500 = yf.download(tickers="^GSPC", start=f"{year}-01-01", end=f"{year}-12-31", progress=False)
    sp500_dr = sp500["Close"].pct_change().dropna()
    
    # 2. Carregando retornos do ano
    try:
        returns = pd.read_parquet(f"../../data/02_clean/returns_new_{year}.parquet")
    except FileNotFoundError:
        print(f"Arquivo de retornos para {year} não encontrado. Pulando...")
        continue

    for k in ks:
        all_betas[year][k] = {}
        
        for centrality in centralities:
            # 3. Pegando metadados dos tickers (ajustado para k variável)
            try:
                metadata_path = f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}_{k}.csv"
                tickers = pd.read_csv(metadata_path)["Ticker"]
                
                # Filtrar apenas tickers que existem no arquivo de retornos
                valid_tickers = [t for t in tickers if t in returns.columns]
                portfolio_returns = returns[valid_tickers].mean(axis=1) # Retorno médio do portfólio
                
                # 4. Calcular Beta
                beta_val = beta_ols(portfolio_returns, sp500_dr)
                all_betas[year][k][centrality] = beta_val
                
            except Exception as e:
                print(f"Erro no portfólio {centrality} (k={k}, ano={year}): {e}")
                all_betas[year][k][centrality] = np.nan

# --- OPCIONAL: Converter resultados para um DataFrame longo (melhor para plotar) ---
rows = []
for year, ks_dict in all_betas.items():
    for k, cents in ks_dict.items():
        for centrality, beta in cents.items():
            rows.append({"Year": year, "k": k, "Centrality": centrality, "Beta": beta})

df_results = pd.DataFrame(rows)
print("\nProcessamento concluído.")
print(df_results.head())

Processando ano: 2015...
Processando ano: 2016...
Processando ano: 2017...
Processando ano: 2018...
Processando ano: 2019...
Processando ano: 2020...
Processando ano: 2021...
Processando ano: 2022...
Processando ano: 2023...
Processando ano: 2024...

Processamento concluído.
   Year   k  Centrality      Beta
0  2015   1     central  1.122735
1  2015   1  peripheral  0.578857
2  2015  10     central  0.980891
3  2015  10  peripheral  0.639678
4  2015  30     central  1.039241


In [ ]:
df_results.to_parquet("../../data/07_portfolio_metadata/beta_df.parquet")

In [1]:
import pandas as pd

years = range(2015, 2024)
returns_dict = {}

for year in years:
    for centrality in ["central", "peripheral"]:
        for k in [1, 10, 30]:
            returns_dict[f"{centrality}_{year}_{k}"] = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date")

In [2]:
import yfinance as yf

momentum_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    first_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year-1}-12-29",
        end=f"{year}-01-01"
    )["Close"]

    last_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year}-12-29",
        end=f"{year+1}-01-01"
    )["Close"]

    momentum_dict[portfolio_name] = last_price.iloc[0] / first_price.iloc[0] - 1

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*****

KeyboardInterrupt: 

In [ ]:
momentum_df = (
    pd.concat(momentum_dict, names=["portfolio", "Ticker"])
    .reset_index(name=f"momentum12mo_{year}")
)
momentum_df.to_parquet(f"../../data/07_portfolios_metadata/momentum_df_{year}.parquet", index=False)

In [30]:
momentum_df

,portfolio,Ticker,momentum12mo_2024
0,central,AAPL,0.316343
1,central,ACWI,0.177692
2,central,ADI,0.088667
3,central,AMAT,0.017571
4,central,AMKR,-0.204909
...,...,...,...
138,peripheral,VIRC,-0.144975
139,peripheral,WFCF,-0.087085
140,peripheral,WKSP,-0.328859
141,peripheral,XOMA,0.410270


## Completo beta e momentum

In [16]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from tqdm import tqdm

df = pd.read_parquet(f"../../data/02_clean/returns_{year-9}_{year}.parquet").dropna(axis=1, how='all').drop(columns=["ABVC", "WKSP"])
tickers = list(df.columns)

def beta_ols(rp, rm):
    joined = rp.to_frame("rp").join(rm, how="inner").dropna()
    if len(joined) < 20:  # not enough observations
        return np.nan
    y = joined["rp"]
    X = sm.add_constant(joined["^GSPC"])
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model.params["^GSPC"]

def momentum_12m(prices):
    """Total return over the year (or log return)."""
    prices = prices.dropna()
    if len(prices) < 2:
        return np.nan
    return (prices.iloc[-1] / prices.iloc[0]) - 1

beta_momentum_df = {}

for year in range(2019, 2025):
    print(f"\n=== {year} ===")
    start, end = f"{year}-01-01", f"{year}-12-31"

    # Market returns
    sp500_prices = yf.download("^GSPC", start=start, end=end, progress=False)["Close"]
    sp500_dr = sp500_prices.pct_change().dropna()
    sp500_dr.name = "rm"

    # Download all tickers at once (much faster than one by one)
    raw = yf.download(tickers, start=start, end=end, progress=True)["Close"]

    records = []
    for ticker in tqdm(tickers, desc=f"Computing {year}"):
        if ticker not in raw.columns:
            continue
        prices = raw[ticker].dropna()
        returns = prices.pct_change().dropna()

        b = beta_ols(returns, sp500_dr)
        m = momentum_12m(prices)

        records.append({
            "Ticker":            ticker,
            f"beta_{year}":      b,
            f"momentum_{year}":  m,
        })

    beta_momentum_df[year] = pd.DataFrame(records)
    print(beta_momentum_df[year].describe())


=== 2019 ===


[*****                 10%                       ]  41 of 404 completed$SPNS: possibly delisted; no price data found  (1d 2019-01-01 -> 2019-12-31) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  404 of 404 completed

1 Failed download:
['SPNS']: possibly delisted; no price data found  (1d 2019-01-01 -> 2019-12-31) (Yahoo error = "No data found, symbol may be delisted")
Computing 2019: 100%|██████████| 404/404 [00:00<00:00, 418.03it/s]


        beta_2019  momentum_2019
count  403.000000     403.000000
mean     0.863532       0.256117
std      0.524210       0.548215
min     -1.041440      -0.930719
25%      0.535024      -0.012655
50%      0.880827       0.208987
75%      1.215908       0.410995
max      2.505274       5.652174

=== 2020 ===


[*****                 10%                       ]  41 of 404 completed$SPNS: possibly delisted; no price data found  (1d 2020-01-01 -> 2020-12-31) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  404 of 404 completed

1 Failed download:
['SPNS']: possibly delisted; no price data found  (1d 2020-01-01 -> 2020-12-31) (Yahoo error = "No data found, symbol may be delisted")
Computing 2020: 100%|██████████| 404/404 [00:00<00:00, 454.75it/s]


        beta_2020  momentum_2020
count  403.000000     403.000000
mean     0.913037       0.363427
std      0.426766       1.165388
min     -1.184414      -0.942130
25%      0.709746      -0.135227
50%      0.967766       0.063333
75%      1.191775       0.400031
max      1.913806       9.769231

=== 2021 ===


...

        beta_2022  momentum_2022
count  403.000000     403.000000
mean     0.877820      -0.165338
std      0.492375       0.369835
min     -0.054619      -0.983227
25%      0.544726      -0.387652
50%      0.818590      -0.173786
75%      1.176722       0.034468
max      2.397115       2.769811

=== 2023 ===


[*****                 10%                       ]  41 of 404 completed$SPNS: possibly delisted; no price data found  (1d 2023-01-01 -> 2023-12-31) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  404 of 404 completed

2 Failed downloads:
['SPNS']: possibly delisted; no price data found  (1d 2023-01-01 -> 2023-12-31) (Yahoo error = "No data found, symbol may be delisted")
['DXLG']: TypeError("'NoneType' object is not subscriptable")
Computing 2023: 100%|██████████| 404/404 [00:01<00:00, 281.90it/s]


        beta_2023  momentum_2023
count  402.000000     402.000000
mean     0.995722       0.164902
std      0.566218       0.673488
min     -0.205251      -0.921630
25%      0.607845      -0.113638
50%      0.941753       0.052514
75%      1.308484       0.299269
max      3.462636       8.460000

=== 2024 ===


[*****                 10%                       ]  41 of 404 completed$SPNS: possibly delisted; no price data found  (1d 2024-01-01 -> 2024-12-31) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  404 of 404 completed

2 Failed downloads:
['SPNS']: possibly delisted; no price data found  (1d 2024-01-01 -> 2024-12-31) (Yahoo error = "No data found, symbol may be delisted")
['CWST']: TypeError("'NoneType' object is not subscriptable")
Computing 2024: 100%|██████████| 404/404 [00:01<00:00, 362.09it/s]

        beta_2024  momentum_2024
count  402.000000     402.000000
mean     0.930785       0.180870
std      0.639705       1.082348
min     -3.196036      -0.905263
25%      0.548338      -0.156719
50%      0.897403       0.063800
75%      1.281409       0.260466
max      3.214299      15.576470


In [20]:
from functools import reduce

combined_df = reduce(
    lambda left, right: pd.merge(left, right, on="Ticker", how="outer"),
    beta_momentum_df.values()
)

combined_df.to_csv("../../data/07_portfolios_metadata/beta_momentum.csv")